In [ ]:
# from databricks.connect import DatabricksSession
# import os
# import config

# os.environ["DATABRICKS_HOST"] = config.DATABRICKS_HOST
# os.environ["DATABRICKS_TOKEN"] = config.DATABRICKS_TOKEN
# os.environ["DATABRICKS_CLUSTER_ID"] = config.DATABRICKS_CLUSTER_ID
# os.environ["DATABRICKS_WORKSPACE_ID"] = config.DATABRICKS_WORKSPACE_ID

# spark = DatabricksSession.builder.getOrCreate()

In [1]:
from pyspark.sql import SparkSession
import config

DATABRICKS_HOST = config.DATABRICKS_HOST
DATABRICKS_TOKEN = config.DATABRICKS_TOKEN
DATABRICKS_CLUSTER_ID = config.DATABRICKS_CLUSTER_ID

# Spark Connect URL format
remote_uri = f"sc://{DATABRICKS_HOST.replace("https://", "")}:443/;token={DATABRICKS_TOKEN};x-databricks-cluster-id={DATABRICKS_CLUSTER_ID}"

# Create SparkSession using Spark Connect
spark = SparkSession.builder.remote(remote_uri).config("spark.connect.session.connectML.enabled", "true").getOrCreate()

In [2]:
print(spark.version)

4.0.0


In [3]:
# Validate Spark Connect
df = spark.range(5)
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [4]:

from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline

import mlflow
from mlflow.models.signature import infer_signature
from mlflow.exceptions import MlflowException
from pyspark.sql.functions import array, col
from datetime import datetime

In [5]:
# Catalog name where the model artifacts will be stored in Unity Catalog
CATALOG_NAME = "alexander_booth" 
SCHEMA_NAME = "default" # Schema (database) name within the catalog to organize model artifacts
TABLE_NAME = "breast_cancer_training_data"
MODEL_NAME = "breast_cancer_model"

In [6]:
# Define configs
input_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
target_col = "label"
uc_model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{MODEL_NAME}"

# Base experiment name
today = datetime.now().strftime("%Y%m%d")  # Format: YYYYMMDD
experiment_name = f"breast_cancer_{today}"

In [7]:
# --- MLflow config ---
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

# Define experiment path (UC)
experiment_path = f"/Users/alexander.booth@databricks.com/{experiment_name}"
artifact_location = "dbfs:/Volumes/alexander_booth/default/mlflow_artifacts"

try:
    mlflow.create_experiment(
        name=experiment_path,
        artifact_location=artifact_location
    )
except MlflowException:
    pass

mlflow.set_experiment(experiment_path)

<Experiment: artifact_location='dbfs:/Volumes/alexander_booth/default/mlflow_artifacts', creation_time=1748969138681, experiment_id='372649807038088', last_update_time=1748981054950, lifecycle_stage='active', name='/Users/alexander.booth@databricks.com/breast_cancer_20250603', tags={'mlflow.experiment.sourceName': '/Users/alexander.booth@databricks.com/breast_cancer_20250603',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'alexander.booth@databricks.com',
 'mlflow.ownerId': '8950113264559567'}>

In [8]:
# --- Load Data ---
df_raw = spark.read.table(input_table)
df_raw = df_raw.dropna()
label_col = "label"
feature_cols = [c for c in df_raw.columns if c != label_col] # Dynamically select feature columns (exclude 'label')

# Train/Test Split
train_df, test_df = df_raw.randomSplit([0.8, 0.2], seed=42)
train_df.show(5)


+-----------+------------+--------------+---------+---------------+----------------+--------------+-------------------+-------------+----------------------+------------+-------------+---------------+----------+----------------+-----------------+---------------+--------------------+--------------+-----------------------+------------+-------------+---------------+----------+----------------+-----------------+---------------+--------------------+--------------+-----------------------+-----+
|mean_radius|mean_texture|mean_perimeter|mean_area|mean_smoothness|mean_compactness|mean_concavity|mean_concave_points|mean_symmetry|mean_fractal_dimension|radius_error|texture_error|perimeter_error|area_error|smoothness_error|compactness_error|concavity_error|concave_points_error|symmetry_error|fractal_dimension_error|worst_radius|worst_texture|worst_perimeter|worst_area|worst_smoothness|worst_compactness|worst_concavity|worst_concave_points|worst_symmetry|worst_fractal_dimension|label|
+-----------+-

In [9]:
import logging

# Silence the log streaming server warnings
logger = logging.getLogger("pyspark.ml.torch")
logger.setLevel(logging.FATAL)
logger.propagate = False  # prevent bubbling up to root logger

In [10]:
# --- MLlib via Spark Connect ---
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")
lr = LogisticRegression(featuresCol="scaled_features")

pipeline = Pipeline(stages=[assembler, scaler, lr])

grid = ParamGridBuilder().addGrid(lr.maxIter, [2, 200]).build()

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=grid,
    parallelism=2,
    evaluator=BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC"),
)

# --- Train + Log ---
now = datetime.now().strftime("%Y%m%d_%H%M%S")  # Format: YYYYMMDD_HHMMSS
run_name = f"logistic-regression-{now}"

with mlflow.start_run(run_name=run_name) as run:
    # mlflow.pyspark.ml.autolog()

    model = cv.fit(train_df)
    predictions = model.transform(test_df)

    # Log metrics
    binary_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
    mlflow.log_metric("areaUnderROC", binary_eval.evaluate(predictions))

    for metric in ["accuracy"]:
        evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName=metric)
        mlflow.log_metric(metric, evaluator.evaluate(predictions))

    # Log best model params
    best_model = model.bestModel.stages[1]  # scaler = stage[0], lr = stage[1]
    for param, value in best_model.extractParamMap().items():
        mlflow.log_param(param.name, value)

    # Sample some input + output rows to infer schema
    sample_input = test_df.limit(5).toPandas()
    sample_output = predictions.limit(5).toPandas()[["prediction"]]

    signature = infer_signature(sample_input, sample_output)

    # Explicitly log the model to UC-compatible location
    mlflow.spark.log_model(model.bestModel, artifact_path="model", signature=signature)

    # Register model
    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"

    print(f"Run ID: {run_id}")
    print(f"Model URI: {model_uri}")

    # Create new model version
    mlflow.register_model(
        model_uri=f"runs:/{run_id}/model",
        name="alexander_booth.default.breast_cancer_model"
    )

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/06/03 15:15:37 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /var/folders/xn/_5vq53hj5ld7v66jgsmc64f40000gp/T/tmp7cjtzy6r, flavor: spark). Fa

Run ID: 2a7c3dc6f22d4010bb0b5abfac0690cd
Model URI: runs:/2a7c3dc6f22d4010bb0b5abfac0690cd/model


Registered model 'alexander_booth.default.breast_cancer_model' already exists. Creating a new version of this model...
Created version '7' of model 'alexander_booth.default.breast_cancer_model'.


🏃 View run logistic-regression-20250603_151503 at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/372649807038088/runs/2a7c3dc6f22d4010bb0b5abfac0690cd
🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/372649807038088


### ⚠️ Why this fails with Spark Connect

This example uses `pyspark.ml` (the classic Spark MLlib API), which depends on the JVM and requires access to a `SparkContext`. 

Because Spark Connect runs Python code in a client-server model **without an embedded JVM**, attempting to use components like `StandardScaler`, `Pipeline`, or `CrossValidator` from `pyspark.ml` results in the following error:

